# Chain-of-Thought (CoT)

*Level 8 — Reasoning Strategies*

## Objective

The baseline every other strategy in this level is measured against: one linear prompt asks the
model to reason step by step, then commit to a final answer, in a single LLM call. No branching,
no backtracking, no evaluator second-guessing itself.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "chain-of-thought"))

from reasoning_common.dataset import prepare, sample_gsm8k
from reasoning_common.embed import OllamaEmbedder
from reasoning_common.llm import OllamaLLM
from reasoning_common.retrieval import DenseRetriever
from cot_prompt import cot_answer

data = prepare()
print(f"corpus: {len(data.corpus)} real StrategyQA facts, {len(data.questions)} labeled questions")

corpus: 282 real StrategyQA facts, 120 labeled questions


## A real question, with real retrieved evidence

In [2]:
embedder = OllamaEmbedder()
llm = OllamaLLM()
retriever = DenseRetriever.from_corpus(data.corpus, embedder=embedder)

question = "Would the top of Mount Fuji stick out of the Sea of Japan?"
retrieved = retriever.search(question, top_k=5)
context = "\n".join(data.corpus[doc_id] for doc_id, _ in retrieved)

print("Retrieved facts:")
for doc_id, score in retrieved:
    print(f"  {score:.3f}  {data.corpus[doc_id]}")

Retrieved facts:
  0.732  The average depth of the Sea of Japan is 5,748 feet (1,752 metres) and its maximum depth is 12,276 feet (3,742 metres) Mount Fuji is 3,776.24 metres (12,389.2 ft) tall.
  0.636  Japan is a country in East Asia.
  0.566  Mount Emei is a 70 ton mountain located in China.
  0.542  RMS Titanic was a British passenger ship.
  0.531  Nissan's headquarters are located in Yokohama, Japan.


In [3]:
result = cot_answer(question, context, llm=llm)
print("Question:", question)
print("\nReasoning:\n", result["reasoning"])
print("\nParsed answer:", result["answer"])
print("LLM calls:", result["llm_calls"])

Question: Would the top of Mount Fuji stick out of the Sea of Japan?

Reasoning:
 To determine if the top of Mount Fuji would stick out of the Sea of Japan, we need to compare its height with the depth of the sea. 

First, let's convert the height of Mount Fuji into feet for consistency: 3,776.24 meters * (1 foot / 0.3048 meters) = approximately 12,389.2 feet.

Now, let's compare this height with the maximum depth of the Sea of Japan: 12,276 feet. Since 12,389.2 feet is greater than 12,276 feet, Mount Fuji's peak would indeed be above the surface of the sea.

Answer: Yes

Parsed answer: True
LLM calls: 1


## What I observed

CoT correctly solved a genuinely non-trivial numeric comparison in one pass: it converted Mount
Fuji's height (3,776.24m) into feet (12,389.2ft), compared it directly against the Sea of Japan's
real maximum depth (12,276ft) from the retrieved evidence, and correctly concluded the mountain's
peak would clear the surface -- matching the real StrategyQA ground truth (`True`) exactly, with
the arithmetic shown and correct. One LLM call, no branching, and it got the hard part (the actual
number comparison) right. Keep this exact question in mind for `02_tree_of_thought.ipynb` and
`03_graph_of_thought.ipynb` -- it is not resolved the same way there.

## Calibration: GSM8K, no retrieval at all

Isolating CoT's raw reasoning quality from retrieval quality entirely -- GSM8K's grade-school
math word problems are self-contained; nothing needs to be retrieved to answer them.

In [4]:
gsm_samples = sample_gsm8k(n=3, seed=42)
for sample in gsm_samples:
    print("Q:", sample["question"][:150])
    print("Real answer:", sample["answer"])
    print()

/Users/yessinezghal/Desktop/learn/rag/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Q: Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.
Real answer: 109

Q: Lorraine and Colleen are trading stickers for buttons. Each large sticker is worth a large button or three small buttons. A small sticker is worth one
Real answer: 89

Q: Indras has 6 letters in her name. Her sister's name has 4 more letters than half of the letters in Indras' name. How many letters are in Indras and he
Real answer: 13



In [5]:
from cot_prompt import COT_PROMPT

# GSM8K needs a numeric answer, not yes/no -- reuse the same linear,
# single-call CoT *shape* with a prompt suited to a numeric final answer,
# rather than reusing cot_answer's yes/no parser.
GSM_COT_PROMPT = (
    "Solve the following math problem. Think through it step by step, "
    "then give your final numeric answer on its own last line, in "
    "exactly this form: 'Answer: <number>'.\n\nProblem: {question}\n\nReasoning:"
)

sample = gsm_samples[0]
raw = llm.complete(GSM_COT_PROMPT.format(question=sample["question"]))
print("Question:", sample["question"])
print("\nModel reasoning:\n", raw)
print("\nReal answer:", sample["answer"])

Question: Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.

Model reasoning:
 To solve this problem, we need to first find the current ages of Darrell and Allen. Let's assume that Darrell's age is 7x and Allen's age is 11x.

We know that their total age now is 162, so we can set up an equation:

7x + 11x = 162

Combine like terms:

18x = 162

Divide both sides by 18:

x = 9

Now that we have the value of x, we can find Darrell's and Allen's current ages:

Darrell's age: 7x = 7(9) = 63
Allen's age: 11x = 11(9) = 99

To find Allen's age 10 years from now, we add 10 to his current age:

Allen's age in 10 years: 99 + 10 = 109

Answer: 109

Real answer: 109


## What I observed (GSM8K calibration)

Also correct: the model set up the ratio (7x : 11x, total 162 -> x=9), solved for both ages
(Darrell 63, Allen 99), and correctly projected Allen's age 10 years forward (109) -- matching the
real GSM8K answer exactly. This confirms the model's raw step-by-step arithmetic reasoning is
genuinely solid on a clean, self-contained problem with no retrieval noise involved at all -- a
useful baseline before judging how it performs once real (sometimes imperfect) retrieved evidence
and a multi-step search are added back in.

## Common Failure Modes anticipated for CoT specifically

- A single linear pass cannot backtrack -- if the very first reasoning step goes wrong (a bad
  arithmetic comparison, a wrong assumption), nothing downstream can recover from it within the
  same call. Tree-of-Thought and Graph-of-Thoughts exist specifically to address this; see
  `02_tree_of_thought.ipynb` and `03_graph_of_thought.ipynb` for whether that actually helps
  here, measured, not assumed.
- CoT's entire reasoning trace is exactly as reliable as the model's raw one-shot judgment --
  there is no independent evaluator step to catch it if it drifts, for better (nothing to add
  noise) or worse (nothing to catch a mistake either).